# Relative-humidity sensitivity

MQ sensors read lower Rs in humid air even with no gas, which is why the fusion engine
damps the gas score when humidity is high and heat is not rising. This notebook measures
how much the Rs/R0 ratio moves per 10 % RH on this unit, from a clean-air capture while
humidity changes (a bathroom after a shower, or a closed box with a damp cloth), and
feeds the RH compensation planned for M2.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))  # run from tools/calibration, next to fxcal.py
import fxcal
import matplotlib.pyplot as plt

In [ ]:
CAPTURE = None  # e.g. "../../captures/rh-sweep-2026-10-03.csv"

cap = fxcal.load_capture(CAPTURE) if CAPTURE else fxcal.demo_rh()
if not CAPTURE:
    print("SYNTHETIC DATA. Set CAPTURE to a bench recording before drawing any conclusion.")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for column in ("mq2_ratio", "mq135_ratio"):
    ts_ratio, ratio = fxcal.series(cap, column)
    ts_rh, rh_values = fxcal.series(cap, "rh_pct")
    by_time = dict(zip(ts_rh, rh_values))
    pairs = [(by_time[t], r) for t, r in zip(ts_ratio, ratio) if t in by_time]
    if len(pairs) < 10:
        print(f"{column}: not enough paired samples")
        continue
    xs, ys = zip(*pairs)
    slope, intercept, r2 = fxcal.linear_fit(list(xs), list(ys))
    print(f"{column}: {slope * 10:+.3f} ratio per 10 % RH (r2 {r2:.2f})")
    ax.scatter(xs, ys, s=6, label=column)
    ax.plot([min(xs), max(xs)], [intercept + slope * min(xs), intercept + slope * max(xs)])
ax.set_xlabel("relative humidity (%)")
ax.set_ylabel("Rs / R0")
ax.legend()
plt.show()